In [ ]:
#!/usr/bin/env sage

from sage.all import *
import itertools
import subprocess
import tempfile
import os
import json
import time
import signal
from datetime import datetime
import numpy as np
import pickle
import shutil
import sys

# Field definitions
SMALL_PRIME = 65537
STANDARD_PRIME1 = 536870923  # ≈2^29, Supports msolve
STANDARD_PRIME2 = 2147483489  # ≈2^31, Supports msolve
LARGE_PRIME = 9223372036854775783  # ≈2^63, no msolve support
CHECKPOINT_VERSION = 1
CHECKPOINT_FILE = os.path.join("out", "benchmark_results_extended_progress.json")

def create_extension_field_2_16():
    """Create GF(2^16) extension field with x^16 + x^5 + x^3 + x^2 + 1"""
    F2 = GF(2)
    R = PolynomialRing(F2, 'x')
    x = R.gen()
    irreducible = x^16 + x^5 + x^3 + x^2 + 1
    return GF(2**16, name='t', modulus=irreducible)

def create_extension_field_2_32():
    """Create GF(2^32) extension field with x^32 + x^15 + x^9 + x^7 + x^4 + x^3 + 1"""
    F2 = GF(2)
    R = PolynomialRing(F2, 'x')
    x = R.gen()
    irreducible = x^32 + x^15 + x^9 + x^7 + x^4 + x^3 + 1
    return GF(2**32, name='t', modulus=irreducible)

def create_extension_field_2_64():
    """Create GF(2^64) extension field with specified irreducible polynomial"""
    F2 = GF(2)
    R = PolynomialRing(F2, 'x')
    x = R.gen()
    # x^64 + x^33 + x^30 + x^26 + x^25 + x^24 + x^23 + x^22 + x^21 + x^20 + x^18 + x^13 + x^12 + x^11 + x^10 + x^7 + x^5 + x^4 + x^2 + x + 1
    irreducible = (x^64 + x^33 + x^30 + x^26 + x^25 + x^24 + x^23 + x^22 + x^21 + x^20 + 
                   x^18 + x^13 + x^12 + x^11 + x^10 + x^7 + x^5 + x^4 + x^2 + x + 1)
    return GF(2**64, name='t', modulus=irreducible)

def create_extension_field_2_128():
    """Create GF(2^128) extension field with x^128 + x^7 + x^2 + x + 1"""
    F2 = GF(2)
    R = PolynomialRing(F2, 'x')
    x = R.gen()
    irreducible = x^128 + x^7 + x^2 + x + 1
    return GF(2**128, name='t', modulus=irreducible)

def get_field_info(field):
    """Extract complete field information"""
    field_info = {}
    if hasattr(field, 'characteristic'):
        field_info['char'] = field.characteristic()
        field_info['degree'] = field.degree() if hasattr(field, 'degree') else 1
        field_info['order'] = field.order() if hasattr(field, 'order') else field.characteristic()
        field_info['is_extension'] = field_info['degree'] > 1
    else:
        # Assume it's a prime
        field_info['char'] = field
        field_info['degree'] = 1
        field_info['order'] = field
        field_info['is_extension'] = False
    return field_info

def random_polynomial_system(n, k, field, m, r):
    """Generate random polynomial system with LEX ordering"""
    R = PolynomialRing(field, n, 'x', order='lex')
    x = R.gens()
    ps = []
    
    all_exponents = list(itertools.product(range(k + 1), repeat=n))
    valid_exponents = [exp for exp in all_exponents if sum(exp) <= k]
    
    r0 = min(r, binomial(n+k, k)) if r > 0 else binomial(n+k, k)
    
    for i in range(m):
        poly = 0
        used_indices = set()
        while len(used_indices) < min(r0, len(valid_exponents)):
            idx = randint(0, len(valid_exponents) - 1)
            if idx not in used_indices:
                exponents = valid_exponents[idx]
                coefficient = field.random_element()
                if coefficient != 0:
                    term = coefficient * prod([x[j] ** exponents[j] for j in range(n)])
                    poly += term
                    used_indices.add(idx)
        ps.append(poly)
    
    return ps, R

def cleanup_all_processes():
    """Cleanup to avoid process conflicts"""
    try:
        os.system("pkill -9 -f magma 2>/dev/null")
        os.system("pkill -9 -f Magma 2>/dev/null")
        os.system("rm -f /tmp/magma_* 2>/dev/null")
        os.system("rm -f /tmp/sage_magma_* 2>/dev/null")
        os.system("rm -f /tmp/interface* 2>/dev/null")
        
        import gc
        gc.collect()
        time.sleep(0.5)
    except:
        pass

def benchmark_magma_subprocess(polys, ring, field_info, timeout=120):
    """Benchmark Magma using subprocess (F4+FGLM)"""
    try:
        nvars = ring.ngens()
        var_names = ','.join(ring.variable_names())
        
        script_lines = []
        
        # Correctly handle extension fields
        if field_info['is_extension']:
            char = field_info['char']
            deg = field_info['degree']
            script_lines.append(f"F<t> := GF({char}, {deg});")
        else:
            char = field_info['char']
            script_lines.append(f"F := GF({char});")
        
        script_lines.extend([
            f"R<{var_names}> := PolynomialRing(F, {nvars}, \"lex\");",
            "polys := [",
        ])
        
        for i, poly in enumerate(polys):
            poly_str = str(poly)
            if i < len(polys) - 1:
                script_lines.append(f"    {poly_str},")
            else:
                script_lines.append(f"    {poly_str}")
        
        script_lines.extend([
            "];",
            "I := ideal<R | polys>;",
            "gb := GroebnerBasis(I);",
            "printf \"%o\\n\", #gb;",
            "quit;"
        ])
        
        with tempfile.NamedTemporaryFile(mode='w', suffix='.m', delete=False) as f:
            f.write('\n'.join(script_lines))
            script_file = f.name
        
        start_time = time.time()
        result = subprocess.run(
            ["magma", script_file],
            capture_output=True,
            text=True,
            timeout=timeout,
            stdin=subprocess.DEVNULL  # FIX: Prevent hanging in command line
        )
        elapsed = time.time() - start_time
        
        os.remove(script_file)
        
        if result.returncode == 0:
            return elapsed
        else:
            return None
            
    except subprocess.TimeoutExpired:
        try:
            os.remove(script_file)
        except:
            pass
        return f">{timeout}"
    except Exception:
        return None

def benchmark_magma_resultant(polys, ring, field_info, timeout=120):
    """Benchmark Magma using Resultant method (only for 2 equations)"""
    if len(polys) != 2:
        return None
        
    try:
        nvars = ring.ngens()
        var_names = ring.variable_names()
        vars_str = ','.join(var_names)
        
        script_lines = []
        
        # Correctly handle extension fields
        if field_info['is_extension']:
            char = field_info['char']
            deg = field_info['degree']
            script_lines.append(f"F<t> := GF({char}, {deg});")
        else:
            char = field_info['char']
            script_lines.append(f"F := GF({char});")
        
        script_lines.extend([
            f"R<{vars_str}> := PolynomialRing(F, {nvars}, \"lex\");",
            f"f := {str(polys[0])};",
            f"g := {str(polys[1])};",
            "",
            "// Compute resultant with respect to last variable",
        ])
        
        if nvars == 2:
            script_lines.extend([
                f"res := Resultant(f, g, {var_names[-1]});",
                "// Solve univariate polynomial",
                "if res ne 0 then",
                "    UP := PolynomialRing(F);",
                "    res_uni := UP!res;",
                "    roots := Roots(res_uni);",
                "    printf \"Found %o roots\\n\", #roots;",
                "else",
                "    printf \"Resultant is zero\\n\";",
                "end if;",
            ])
        else:
            script_lines.extend([
                f"// Eliminate {var_names[-1]} first",
                f"res1 := Resultant(f, g, {var_names[-1]});",
                "if res1 ne 0 then",
                "    printf \"Resultant computed\\n\";",
                "else",
                "    printf \"Resultant is zero\\n\";",
                "end if;",
            ])
        
        script_lines.append("quit;")
        
        with tempfile.NamedTemporaryFile(mode='w', suffix='.m', delete=False) as f:
            f.write('\n'.join(script_lines))
            script_file = f.name
        
        start_time = time.time()
        result = subprocess.run(
            ["magma", script_file],
            capture_output=True,
            text=True,
            timeout=timeout,
            stdin=subprocess.DEVNULL  # FIX: Prevent hanging in command line
        )
        elapsed = time.time() - start_time
        
        os.remove(script_file)
        
        if result.returncode == 0:
            return elapsed
        else:
            return None
            
    except subprocess.TimeoutExpired:
        try:
            os.remove(script_file)
        except:
            pass
        return f">{timeout}"
    except Exception:
        return None

def benchmark_drsolve(polys, ring, field_info, m, timeout=120):
    """Benchmark drsolve method - supports extension fields"""
    if m <= 1:
        return None
    
    if not shutil.which("drsolve"):
        return None
    
    try:
        var_names = [str(v) for v in ring.gens()]
        
        with tempfile.NamedTemporaryFile(mode='w', suffix='.dat', delete=False) as f:
            eliminate_vars = var_names[:m-1]
            f.write(",".join(eliminate_vars) + "\n")
            
            # drsolve supports extension fields with p^k format
            if field_info['is_extension']:
                # Write as p^k (e.g., 2^128 for GF(2^128))
                f.write(f"{field_info['char']}^{field_info['degree']}\n")
            else:
                # Write as prime for prime fields
                f.write(f"{field_info['char']}\n")
            
            poly_strs = [str(poly) for poly in polys]
            f.write(", ".join(poly_strs) + "\n")
            input_file = f.name
        
        start = time.time()
        result = subprocess.run(
            ["drsolve", "--silent", input_file],
            capture_output=True,
            text=True,
            timeout=timeout,
            stdin=subprocess.DEVNULL  # FIX: Prevent hanging in command line
        )
        elapsed = time.time() - start
        
        try:
            os.remove(input_file)
        except:
            pass
        
        if result.returncode == 0:
            return elapsed
        else:
            if result.stderr:
                print(f"drsolve error: {result.stderr[:200]}")
            return None
            
    except subprocess.TimeoutExpired:
        try:
            os.remove(input_file)
        except:
            pass
        return f">{timeout}"
    except Exception as e:
        print(f"drsolve exception: {str(e)[:200]}")
        return None

def benchmark_msolve(polys, ring, field_info, n, m, timeout=120):
    """Benchmark msolve elimination (only for supported fields)"""
    if n <= 1 or field_info['char'] > (2**31 - 1):
        return None
    
    # msolve only supports prime fields, not extension fields
    if field_info['is_extension']:
        return None
        
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.ms', delete=False) as f:
            var_names = [f"x{i}" for i in range(n)]
            f.write(", ".join(var_names) + "\n")
            f.write(f"{field_info['char']}\n")
            
            for i, poly in enumerate(polys):
                poly_str = str(poly)
                if i < len(polys) - 1:
                    f.write(poly_str + ",\n")
                else:
                    f.write(poly_str + "\n")
            input_file = f.name
        
        eliminate_count = min(n-1, m-1) if m > 1 else n-1
        
        start = time.time()
        result = subprocess.run(
            ["msolve", "-e", str(eliminate_count), "-g", "2", "-f", input_file],
            capture_output=True,
            text=True,
            timeout=timeout,
            stdin=subprocess.DEVNULL  # FIX: Prevent hanging in command line
        )
        elapsed = time.time() - start
        
        os.remove(input_file)
        
        if result.returncode == 0:
            return elapsed
        else:
            return None
            
    except subprocess.TimeoutExpired:
        try:
            os.remove(input_file)
        except:
            pass
        return f">{timeout}"
    except Exception:
        return None

def run_single_benchmark(n, d, field, field_info, m, r, timeout=120, 
                        include_resultant=False, 
                        include_msolve=False,
                        skip_solvers=None):
    """Run a single benchmark test
    
    Args:
        skip_solvers: set of solver names to skip (will return '>3600' for these)
    """
    if skip_solvers is None:
        skip_solvers = set()
    
    set_random_seed(42 + n + d + m)
    polys, ring = random_polynomial_system(n, d, field, m, r)
    
    results = {}
    
    # Benchmark each method (skip if already timed out at lower degree)
    if 'magma' in skip_solvers:
        results['magma'] = ">3600"
    else:
        results['magma'] = benchmark_magma_subprocess(polys, ring, field_info, timeout)
    
    if 'drsolve' in skip_solvers:
        results['drsolve'] = ">3600"
    else:
        results['drsolve'] = benchmark_drsolve(polys, ring, field_info, m, timeout)
    
    if include_msolve and not field_info['is_extension']:
        if 'msolve' in skip_solvers:
            results['msolve'] = ">3600"
        else:
            results['msolve'] = benchmark_msolve(polys, ring, field_info, n, m, timeout)
    else:
        results['msolve'] = None
    
    if include_resultant and m == 2:
        if 'magma_resultant' in skip_solvers:
            results['magma_resultant'] = ">3600"
        else:
            results['magma_resultant'] = benchmark_magma_resultant(polys, ring, field_info, timeout)
    else:
        results['magma_resultant'] = None
    
    # Calculate empirical time ratios
    if results['magma'] and results['drsolve'] and not isinstance(results['magma'], str) and not isinstance(results['drsolve'], str):
        results['drsolve_magma_time_ratio'] = results['magma'] / results['drsolve']
    else:
        results['drsolve_magma_time_ratio'] = None
    
    return results

def format_time_result(result):
    """Format timing result for display"""
    if result is None:
        return "FAIL"
    elif isinstance(result, str) and result.startswith(">"):
        return result + "s"
    else:
        return f"{result:.4f}s"

def convert_for_json(obj):
    """Convert non-JSON-serializable objects"""
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    elif hasattr(obj, 'parent'):  # Sage types
        return float(obj)
    return obj

def deep_convert(obj):
    """Recursively convert values into JSON-serializable types"""
    if isinstance(obj, dict):
        return {k: deep_convert(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [deep_convert(v) for v in obj]
    else:
        return convert_for_json(obj)

def atomic_write_json(path, data):
    """Atomically write JSON data to disk"""
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.tmp', dir=directory or '.', delete=False) as f:
        json.dump(deep_convert(data), f, indent=2)
        temp_path = f.name
    
    os.replace(temp_path, path)

def atomic_write_pickle(path, data):
    """Atomically write pickle data to disk"""
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    
    with tempfile.NamedTemporaryFile(mode='wb', suffix='.tmp', dir=directory or '.', delete=False) as f:
        pickle.dump(data, f)
        temp_path = f.name
    
    os.replace(temp_path, path)

def get_degree_range(n):
    """Return degree range for a given variable count"""
    if n == 2:
        return range(2, 66, 8)
    elif n == 3:
        return range(2, 22, 2)
    elif n == 4:
        return range(2, 11, 1)
    elif n == 5:
        return range(2, 7, 1)
    elif n == 6:
        return range(2, 5, 1)
    elif n == 7:
        return range(2, 4, 1)
    else:  # n == 8
        return range(2, 3, 1)

def get_benchmark_configs():
    """Return all benchmark configurations"""
    configs = []
    
    for n in [2, 3, 4, 5, 6, 7, 8, 9]:
        configs.append({
            'key': f'fixed_vars_{n}',
            'title': f"Testing with n={n} variables (square system m=n={n})",
            'n': n,
            'm': n,
            'd_range': get_degree_range(n)
        })
    
    configs.extend([
        {
            'key': 'more_vars_n3_m2',
            'title': "Testing with n=3 variables, m=2 equations",
            'n': 3,
            'm': 2,
            'd_range': range(2, 21, 4)
        },
        {
            'key': 'more_vars_n4_m3',
            'title': "Testing with n=4 variables, m=3 equations",
            'n': 4,
            'm': 3,
            'd_range': range(2, 9, 2)
        }
    ])
    
    return configs

def update_skip_solvers(skip_solvers, results):
    """Update skip list based on timeout results"""
    for solver in ['magma', 'drsolve', 'msolve', 'magma_resultant']:
        result = results.get(solver)
        if result is not None and isinstance(result, str) and result.startswith(">"):
            skip_solvers.add(solver)

def count_completed_tests(all_results):
    """Count completed benchmark points stored in checkpoint"""
    total = 0
    for field_results in all_results.values():
        for test_data in field_results.values():
            total += len(test_data)
    return total

def load_checkpoint(checkpoint_file):
    """Load progress checkpoint if it exists"""
    if not os.path.exists(checkpoint_file):
        return None
    
    with open(checkpoint_file, 'r') as f:
        checkpoint = json.load(f)
    
    if checkpoint.get('version') != CHECKPOINT_VERSION:
        raise ValueError(
            f"Unsupported checkpoint version: {checkpoint.get('version')} "
            f"(expected {CHECKPOINT_VERSION})"
        )
    
    return checkpoint

def save_checkpoint(checkpoint_file, run_timestamp, all_results, field_names, field_infos, completed=False):
    """Persist current benchmark progress"""
    checkpoint_data = {
        'version': CHECKPOINT_VERSION,
        'run_timestamp': run_timestamp,
        'updated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'completed': completed,
        'results': all_results,
        'field_names': field_names,
        'field_infos': field_infos
    }
    atomic_write_json(checkpoint_file, checkpoint_data)

def save_final_results(run_timestamp, all_results, field_names, field_infos):
    """Save final benchmark outputs in the same structure as before"""
    payload = {
        'results': all_results,
        'field_names': field_names,
        'field_infos': field_infos,
        'timestamp': run_timestamp
    }
    
    pickle_file = f'benchmark_results_extended_{run_timestamp}.pkl'
    atomic_write_pickle(pickle_file, payload)
    print(f"\n✓ Results saved to {pickle_file}")
    
    json_file = f'benchmark_results_extended_{run_timestamp}.json'
    atomic_write_json(json_file, payload)
    print(f"✓ Results also saved as JSON to {json_file}")
    
    return pickle_file, json_file

def run_all_benchmarks(checkpoint_file=CHECKPOINT_FILE):
    """Run complete benchmark suite and save results"""
    import time
    start_time = time.time()
    
    print("="*100)
    print("POLYNOMIAL SOLVER BENCHMARK - DATA COLLECTION")
    print("="*100)
    print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*100)
    
    # Define fields
    small_field = GF(SMALL_PRIME)
    standard_field1 = GF(STANDARD_PRIME1)
    standard_field2 = GF(STANDARD_PRIME2)
    large_prime_field = GF(LARGE_PRIME)
    
    # Create extension fields with specified moduli
    extension_field_16 = create_extension_field_2_16()
    extension_field_32 = create_extension_field_2_32()
    extension_field_64 = create_extension_field_2_64()
    extension_field_128 = create_extension_field_2_128()
    
    fields = {
        'small': small_field,
        'standard1': standard_field1,
        'standard2': standard_field2,
        'large_prime': large_prime_field,
        #'extension_2_16': extension_field_16,
        'extension_2_32': extension_field_32,
        #'extension_2_64': extension_field_64,
        'extension_2_128': extension_field_128
    }
    
    field_names = {
        'small': f'GF({SMALL_PRIME})',
        'standard1': f'GF({STANDARD_PRIME1})',
        'standard2': f'GF({STANDARD_PRIME2})',
        'large_prime': f'GF({LARGE_PRIME})',
        #'extension_2_16': 'GF(2^16)',
        'extension_2_32': 'GF(2^32)',
        #'extension_2_64': 'GF(2^64)',
        'extension_2_128': 'GF(2^128)'
    }
    
    # Get field information
    field_infos = {key: get_field_info(field) for key, field in fields.items()}
    
    checkpoint = load_checkpoint(checkpoint_file)
    if checkpoint is not None:
        run_timestamp = checkpoint.get('run_timestamp')
        all_results = checkpoint.get('results', {})
        completed_tests = count_completed_tests(all_results)
        print(f"Resuming from checkpoint: {checkpoint_file}")
        print(f"Checkpoint timestamp: {run_timestamp}")
        print(f"Completed benchmark points: {completed_tests}")
    else:
        run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        all_results = {}
        save_checkpoint(checkpoint_file, run_timestamp, all_results, field_names, field_infos, completed=False)
        print(f"Starting new benchmark run")
        print(f"Checkpoint file: {checkpoint_file}")
    
    for field_key, field in fields.items():
        field_info = field_infos[field_key]
        
        print(f"\n{'='*80}")
        print(f"TESTING ON {field_names[field_key]}")
        if field_info['is_extension']:
            print(f"  (Extension field: char={field_info['char']}, degree={field_info['degree']}, order={field_info['order']})")
        print("="*80)
        
        include_msolve = (field_key in ['small', 'standard1', 'standard2'])
        field_results = all_results.setdefault(field_key, {})
        
        for config in get_benchmark_configs():
            n = config['n']
            m = config['m']
            config_key = config['key']
            
            print(f"\n--- {config['title']} ---")
            header = f"{'d':<5} {'Magma':<12} {'drsolve':<12} {'msolve':<12} {'MagmaResultant':<16} {'M/D Ratio':<10}"
            print(header)
            print("-" * len(header))
            
            d_range = config['d_range']
            
            skip_solvers = set()
            existing_test_data = field_results.get(config_key, [])
            existing_by_d = {entry['d']: entry for entry in existing_test_data}
            
            test_data = []
            for d in d_range:
                if d in existing_by_d:
                    entry = existing_by_d[d]
                    results = entry['results']
                    test_data.append(entry)
                    update_skip_solvers(skip_solvers, results)
                else:
                    r = min(300, binomial(n+d, d))
                    timeout = 3600 # min(3600, 300 * d * n)
                    
                    results = run_single_benchmark(
                        n, d, field, field_info, m, r, timeout,
                        include_resultant=(m==2),
                        include_msolve=include_msolve,
                        skip_solvers=skip_solvers
                    )
                    update_skip_solvers(skip_solvers, results)
                    
                    test_data.append({
                        'n': n, 'd': d, 'm': m,
                        'results': results
                    })
                    field_results[config_key] = test_data
                    save_checkpoint(
                        checkpoint_file,
                        run_timestamp,
                        all_results,
                        field_names,
                        field_infos,
                        completed=False
                    )
                
                magma_str = format_time_result(results.get('magma'))
                drsolve_str = format_time_result(results.get('drsolve'))
                msolve_str = format_time_result(results.get('msolve')) if include_msolve else "N/A"
                resultant_str = format_time_result(results.get('magma_resultant')) if m == 2 else "N/A"
                
                ratio_str = "N/A"
                if results.get('drsolve_magma_time_ratio') is not None:
                    ratio_str = f"{results['drsolve_magma_time_ratio']:.4f}"
                
                print(f"{d:<5} {magma_str:<12} {drsolve_str:<12} {msolve_str:<12} {resultant_str:<16} {ratio_str:<10}")
            
            field_results[config_key] = test_data
            save_checkpoint(
                checkpoint_file,
                run_timestamp,
                all_results,
                field_names,
                field_infos,
                completed=False
            )
    
    save_final_results(run_timestamp, all_results, field_names, field_infos)
    save_checkpoint(
        checkpoint_file,
        run_timestamp,
        all_results,
        field_names,
        field_infos,
        completed=True
    )
    
    print("\n" + "="*100)
    print("DATA COLLECTION COMPLETE")
    print("="*100)
    
    # Calculate and display total time
    end_time = time.time()
    total_time = end_time - start_time
    hours = int(total_time // 3600)
    minutes = int((total_time % 3600) // 60)
    seconds = total_time % 60
    
    print(f"End Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Total Time: {hours}h {minutes}m {seconds:.1f}s")
    print("="*100)
    
    return all_results, field_names, run_timestamp

if __name__ == "__main__":
    print("\n" + "="*100)
    print("Make sure magma, msolve and drsolve is installed and in your PATH")
    print("="*100 + "\n")

    # Run benchmarks and save results
    results, names, timestamp = run_all_benchmarks() 
    print(f"\nBenchmark completed at {timestamp}")


In [ ]:
#!/usr/bin/env python3
"""
Visualization script for polynomial solver benchmarks.
Run this after the benchmark completes to generate comparison charts.
"""

import glob
import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np


plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif', 'Georgia']
plt.rcParams['font.size'] = 26
plt.rcParams['axes.labelsize'] = 28
plt.rcParams['axes.titlesize'] = 30
plt.rcParams['xtick.labelsize'] = 26
plt.rcParams['ytick.labelsize'] = 26
plt.rcParams['legend.fontsize'] = 23


FIELD_DISPLAY_MAP = {
    'small': '$F_{65537}$',
    'standard1': '$F_{2^{29}+11}$',
    'standard2': '$F_{2^{31}-159}$',
    'large_prime': '$F_{2^{61}-1}$',
    'extension': '$F_{2^8}$',
}

FIELD_PAIR = ['standard1', 'standard2']

METHOD_STYLES = {
    'drsolve': ('DRSolve', 's-', '#ff7f0e'),
    'magma': ('Magma', 'o-', '#1f77b4'),
    'msolve': ('msolve', '^-', '#2ca02c'),
    'magma_resultant': ('Magma Resultant', 'd-', '#9467bd'),
    'fermat': ('Fermat', 'x-', '#d62728'),
}


def load_fermat_times(path='fermat_timings_recorded.txt'):
    """Read Fermat's n*d timings; failed cases remain unavailable."""
    timings = {}
    fermat_file = Path(path)
    if not fermat_file.exists():
        return timings
    lines = fermat_file.read_text().splitlines()
    for index, line in enumerate(lines[:-1]):
        if not line.strip().startswith('Running '):
            continue
        case = line.strip()[len('Running '):].split('...', 1)[0]
        try:
            n_text, d_text = case.split('x', 1)
            key = (int(n_text), int(d_text))
        except ValueError:
            continue
        result = lines[index + 1].strip()
        timings[key] = None
        if result.startswith('ok, time=') and result.endswith('s'):
            try:
                timings[key] = float(result[len('ok, time='):-1])
            except ValueError:
                pass
    return timings


def load_latest_results():
    """Load the most recent benchmark results file by modification time."""
    candidates = glob.glob('benchmark_results_*.pkl') + glob.glob('benchmark_results_*.json')
    if not candidates:
        raise FileNotFoundError('No benchmark results found!')

    latest_file = max(candidates, key=lambda path: Path(path).stat().st_mtime)
    print(f'Loading from {latest_file}...')

    if latest_file.endswith('.json'):
        with open(latest_file, 'r') as handle:
            return json.load(handle)

    with open(latest_file, 'rb') as handle:
        return pickle.load(handle)


def get_field_names(data):
    """Return field names even when the results file omits field_names."""
    stored_names = data.get('field_names', {})
    return {
        field_key: stored_names.get(field_key, FIELD_DISPLAY_MAP.get(field_key, field_key))
        for field_key in data.get('results', {})
    }


def parse_time(value):
    """Parse a timing value, ignoring failures and timeout markers."""
    if value is None or value == 'FAIL':
        return None

    if isinstance(value, str):
        if value.startswith('>'):
            return None
        try:
            return float(value)
        except ValueError:
            return None

    return float(value)


def get_solver_time(results, solver_key):
    """Read solver timings with backward compatibility for legacy dixon keys."""
    aliases = {
        'drsolve': ['drsolve', 'dixon'],
        'magma': ['magma'],
        'msolve': ['msolve'],
        'magma_resultant': ['magma_resultant'],
    }

    for key in aliases.get(solver_key, [solver_key]):
        if key in results:
            return parse_time(results.get(key))
    return None


def get_speedup_ratio(results):
    """Return Magma/solver ratio for current and legacy benchmark outputs."""
    ratio = results.get('drsolve_magma_time_ratio')
    if ratio is None:
        ratio = results.get('dixon_magma_time_ratio')
    return ratio if isinstance(ratio, (int, float)) else None


def format_log_axis(ax, y_values):
    """Format a base-2 log axis with sparse power-of-two labels."""

    def log2_formatter(x, _pos):
        if x <= 0:
            return '0'
        log2_val = np.log2(x)
        if abs(log2_val - round(log2_val)) < 0.01:
            return f'$2^{{{int(round(log2_val))}}}$'
        return ''

    ax.yaxis.set_major_formatter(ticker.FuncFormatter(log2_formatter))

    valid_y = [y for y in y_values if y is not None and y > 0]
    if not valid_y:
        return

    y_min, y_max = min(valid_y), max(valid_y)
    log_min = int(np.floor(np.log2(y_min)))
    log_max = int(np.ceil(np.log2(y_max)))
    step = max(2, (log_max - log_min) // 6)
    ax.set_yticks([2**power for power in range(log_min, log_max + 1, step)])


def set_data_xticks(ax, x_values):
    """Show only actual data-point ticks on the x-axis."""
    unique_x = sorted(set(x_values))
    ax.set_xticks(unique_x)
    ax.set_xticklabels([str(int(x)) for x in unique_x])


def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(filename, dpi=150, bbox_inches='tight')
    print(f'  Saved: {filename}')
    plt.close(fig)


def plot_log_series(ax, x_values, series_list):
    """Plot benchmark series on a single log-scale time axis."""
    all_y_values = []

    for label, times, style, color in series_list:
        valid_data = [(x, t) for x, t in zip(x_values, times) if t is not None]
        if not valid_data:
            continue

        x_vals, y_vals = zip(*valid_data)
        all_y_values.extend(y_vals)
        ax.plot(x_vals, y_vals, style, label=label, color=color, linewidth=3, markersize=10)

    return all_y_values


def plot_field_comparison(data, field_key, field_name, output_dir='plots'):
    """Create comparison plots for a single field across all n values."""
    Path(output_dir).mkdir(exist_ok=True)

    field_results = data['results'][field_key]
    fermat_times = load_fermat_times() if field_key == 'standard1' else {}
    n_values = sorted(
        int(key.split('_')[-1])
        for key in field_results.keys()
        if key.startswith('fixed_vars_')
    )
    display_name = FIELD_DISPLAY_MAP.get(field_key, field_name)

    for n_value in n_values:
        test_key = f'fixed_vars_{n_value}'
        if test_key not in field_results:
            continue

        degrees = []
        drsolve_times = []
        magma_times = []
        msolve_times = []
        resultant_times = []
        fermat_series = []

        for entry in field_results[test_key]:
            results = entry['results']
            degrees.append(entry['d'])
            drsolve_times.append(get_solver_time(results, 'drsolve'))
            magma_times.append(get_solver_time(results, 'magma'))
            msolve_times.append(get_solver_time(results, 'msolve'))
            resultant_times.append(get_solver_time(results, 'magma_resultant'))
            fermat_series.append(fermat_times.get((n_value, entry['d'])))

        fig, ax = plt.subplots(figsize=(10, 8))

        ax.set_title(f'{display_name}: $n={n_value}$', fontweight='bold')
        ax.set_xlabel('Degree (d)')
        ax.set_ylabel('Time (seconds)')
        ax.set_yscale('log', base=2)
        ax.grid(True, alpha=0.3, which='both', linewidth=0.8)
        ax.grid(True, alpha=0.15, which='minor', linewidth=0.5)

        all_y_values = plot_log_series(
            ax,
            degrees,
            [
                (METHOD_STYLES['drsolve'][0], drsolve_times, METHOD_STYLES['drsolve'][1], METHOD_STYLES['drsolve'][2]),
                (METHOD_STYLES['magma'][0], magma_times, METHOD_STYLES['magma'][1], METHOD_STYLES['magma'][2]),
                (METHOD_STYLES['msolve'][0], msolve_times, METHOD_STYLES['msolve'][1], METHOD_STYLES['msolve'][2]),
                (METHOD_STYLES['magma_resultant'][0], resultant_times, METHOD_STYLES['magma_resultant'][1], METHOD_STYLES['magma_resultant'][2]),
                (METHOD_STYLES['fermat'][0], fermat_series, METHOD_STYLES['fermat'][1], METHOD_STYLES['fermat'][2]),
            ],
        )

        set_data_xticks(ax, degrees)
        format_log_axis(ax, all_y_values)
        ax.legend(loc='best', framealpha=0.95)

        save_figure(fig, f'{output_dir}/{field_key}_n{n_value}_comparison.png')


def plot_cross_field_comparison(data, n_value, output_dir='plots'):
    """Compare standard1 and standard2 for a fixed number of variables."""
    Path(output_dir).mkdir(exist_ok=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = {
        ('standard1', 'drsolve'): '#ff7f0e',
        ('standard1', 'magma'): '#1f77b4',
        ('standard1', 'msolve'): '#2ca02c',
        ('standard2', 'drsolve'): '#9467bd',
        ('standard2', 'magma'): '#d62728',
        ('standard2', 'msolve'): '#8c564b',
    }
    markers = {'drsolve': 's', 'magma': 'o', 'msolve': '^'}
    linestyles = {'standard1': '-', 'standard2': '--'}

    all_degrees = set()
    all_y_values = []

    for field_key in FIELD_PAIR:
        field_results = data['results'].get(field_key, {})
        test_key = f'fixed_vars_{n_value}'
        if test_key not in field_results:
            continue

        for method in ['drsolve', 'magma', 'msolve']:
            degrees = []
            times = []

            for entry in field_results[test_key]:
                time_value = get_solver_time(entry['results'], method)
                if time_value is None:
                    continue
                degrees.append(entry['d'])
                times.append(time_value)
                all_degrees.add(entry['d'])
                all_y_values.append(time_value)

            if not times:
                continue

            label = f"{FIELD_DISPLAY_MAP.get(field_key, field_key)} - {METHOD_STYLES[method][0]}"
            style = markers[method] + linestyles[field_key]
            color = colors[(field_key, method)]
            ax.plot(degrees, times, style, label=label, color=color, linewidth=3, markersize=9, alpha=0.85)

        if field_key == 'standard1' and method == 'drsolve':
            fermat_times = load_fermat_times()
            fermat_degrees = [entry['d'] for entry in field_results[test_key] if fermat_times.get((n_value, entry['d'])) is not None]
            fermat_values = [fermat_times[(n_value, degree_value)] for degree_value in fermat_degrees]
            if fermat_values:
                all_degrees.update(fermat_degrees)
                all_y_values.extend(fermat_values)
                ax.plot(fermat_degrees, fermat_values, METHOD_STYLES['fermat'][1], label=f"{FIELD_DISPLAY_MAP['standard1']} - {METHOD_STYLES['fermat'][0]}", color=METHOD_STYLES['fermat'][2], linewidth=3, markersize=9, alpha=0.85)

    ax.set_title(f'$n={n_value}$', fontweight='bold')
    ax.set_xlabel('Degree (d)')
    ax.set_ylabel('Time (seconds)')
    ax.set_yscale('log', base=2)
    ax.grid(True, alpha=0.3, which='both', linewidth=0.8)
    ax.grid(True, alpha=0.15, which='minor', linewidth=0.5)

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        split = len(handles) // 2
        ax.legend(handles[:split], labels[:split], loc='upper left', framealpha=0.95)
        ax.add_artist(ax.legend(handles[split:], labels[split:], loc='lower right', framealpha=0.95))

    if all_degrees:
        set_data_xticks(ax, sorted(all_degrees))
    format_log_axis(ax, all_y_values)

    save_figure(fig, f'{output_dir}/standard_fields_n{n_value}_comparison.png')


def plot_speedup_ratios(data, output_dir='plots'):
    """Plot Magma/drsolve speedup ratios across all configurations."""
    Path(output_dir).mkdir(exist_ok=True)

    fig, axes = plt.subplots(2, 3, figsize=(24, 14))
    axes = axes.flatten()

    for idx, field_key in enumerate(['small', 'standard1', 'standard2', 'large_prime', 'extension']):
        if idx >= len(axes):
            break

        ax = axes[idx]
        if field_key not in data['results']:
            ax.set_visible(False)
            continue

        field_results = data['results'][field_key]
        display_name = FIELD_DISPLAY_MAP.get(field_key, field_key)
        n_values = sorted(
            int(key.split('_')[-1])
            for key in field_results.keys()
            if key.startswith('fixed_vars_')
        )

        all_degrees = set()
        for n_value in n_values:
            test_key = f'fixed_vars_{n_value}'
            if test_key not in field_results:
                continue

            degrees = []
            ratios = []
            for entry in field_results[test_key]:
                ratio = get_speedup_ratio(entry['results'])
                if ratio is None:
                    continue
                degrees.append(entry['d'])
                ratios.append(ratio)
                all_degrees.add(entry['d'])

            if ratios:
                ax.plot(degrees, ratios, 'o-', label=f'$n={n_value}$', linewidth=3, markersize=10)

        ax.set_title(display_name, fontweight='bold')
        ax.set_xlabel('Degree (d)')
        ax.set_ylabel('Speedup Ratio (Magma/drsolve)')
        ax.axhline(y=1, color='red', linestyle='--', alpha=0.5, linewidth=2.5, label='Equal performance')
        ax.grid(True, alpha=0.3, linewidth=0.8)
        ax.legend(loc='best', framealpha=0.95)
        ax.set_ylim(bottom=0)

        if all_degrees:
            set_data_xticks(ax, sorted(all_degrees))

    if len(axes) > 5:
        axes[5].set_visible(False)

    save_figure(fig, f'{output_dir}/speedup_ratios_all_fields.png')


def plot_fixed_degree_comparison(data, degree=2, output_dir='plots'):
    """Create per-field plots for fixed degree while varying n."""
    Path(output_dir).mkdir(exist_ok=True)
    field_names = get_field_names(data)

    for field_key, field_name in field_names.items():
        if field_key not in data['results']:
            continue

        field_results = data['results'][field_key]
        display_name = FIELD_DISPLAY_MAP.get(field_key, field_name)
        fermat_times = load_fermat_times() if field_key == 'standard1' else {}

        n_values = []
        drsolve_times = []
        magma_times = []
        msolve_times = []
        fermat_series = []

        all_n = sorted(
            int(key.split('_')[-1])
            for key in field_results.keys()
            if key.startswith('fixed_vars_')
        )

        for n_value in all_n:
            test_key = f'fixed_vars_{n_value}'
            if test_key not in field_results:
                continue

            for entry in field_results[test_key]:
                if entry['d'] != degree:
                    continue

                results = entry['results']
                drsolve_time = get_solver_time(results, 'drsolve')
                magma_time = get_solver_time(results, 'magma')
                msolve_time = get_solver_time(results, 'msolve')
                fermat_time = fermat_times.get((n_value, degree))
                if any(value is not None for value in [drsolve_time, magma_time, msolve_time, fermat_time]):
                    n_values.append(n_value)
                    drsolve_times.append(drsolve_time)
                    magma_times.append(magma_time)
                    msolve_times.append(msolve_time)
                    fermat_series.append(fermat_time)
                break

        if not n_values:
            continue

        fig, ax = plt.subplots(figsize=(10, 8))

        ax.set_title(f'{display_name}: $d={degree}$', fontweight='bold')
        ax.set_xlabel('Number of Variables (n)')
        ax.set_ylabel('Time (seconds)')
        ax.set_yscale('log', base=2)
        ax.grid(True, alpha=0.3, which='both', linewidth=0.8)
        ax.grid(True, alpha=0.15, which='minor', linewidth=0.5)

        all_y_values = plot_log_series(
            ax,
            n_values,
            [
                (METHOD_STYLES['drsolve'][0], drsolve_times, METHOD_STYLES['drsolve'][1], METHOD_STYLES['drsolve'][2]),
                (METHOD_STYLES['magma'][0], magma_times, METHOD_STYLES['magma'][1], METHOD_STYLES['magma'][2]),
                (METHOD_STYLES['msolve'][0], msolve_times, METHOD_STYLES['msolve'][1], METHOD_STYLES['msolve'][2]),
                (METHOD_STYLES['fermat'][0], fermat_series, METHOD_STYLES['fermat'][1], METHOD_STYLES['fermat'][2]),
            ],
        )

        set_data_xticks(ax, n_values)
        format_log_axis(ax, all_y_values)
        ax.legend(loc='best', framealpha=0.95)

        save_figure(fig, f'{output_dir}/{field_key}_d{degree}_vary_n.png')


def plot_cross_field_fixed_degree(data, degree=2, output_dir='plots'):
    """Compare standard1 and standard2 for fixed degree while varying n."""
    Path(output_dir).mkdir(exist_ok=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = {
        ('standard1', 'drsolve'): '#ff7f0e',
        ('standard1', 'magma'): '#1f77b4',
        ('standard1', 'msolve'): '#2ca02c',
        ('standard2', 'drsolve'): '#9467bd',
        ('standard2', 'magma'): '#d62728',
        ('standard2', 'msolve'): '#8c564b',
    }
    markers = {'drsolve': 's', 'magma': 'o', 'msolve': '^'}
    linestyles = {'standard1': '-', 'standard2': '--'}

    all_n_values = set()
    all_y_values = []

    for field_key in FIELD_PAIR:
        field_results = data['results'].get(field_key, {})
        all_n = sorted(
            int(key.split('_')[-1])
            for key in field_results.keys()
            if key.startswith('fixed_vars_')
        )

        for method in ['drsolve', 'magma', 'msolve']:
            x_values = []
            y_values = []

            for n_value in all_n:
                test_key = f'fixed_vars_{n_value}'
                if test_key not in field_results:
                    continue

                for entry in field_results[test_key]:
                    if entry['d'] != degree:
                        continue
                    time_value = get_solver_time(entry['results'], method)
                    if time_value is not None:
                        x_values.append(n_value)
                        y_values.append(time_value)
                        all_n_values.add(n_value)
                        all_y_values.append(time_value)
                    break

            if not y_values:
                continue

            label = f"{FIELD_DISPLAY_MAP.get(field_key, field_key)} - {METHOD_STYLES[method][0]}"
            style = markers[method] + linestyles[field_key]
            color = colors[(field_key, method)]
            ax.plot(x_values, y_values, style, label=label, color=color, linewidth=3, markersize=9, alpha=0.85)

        if field_key == 'standard1' and method == 'drsolve':
            fermat_times = load_fermat_times()
            fermat_n_values = []
            fermat_values = []
            for n_value in all_n:
                fermat_time = fermat_times.get((n_value, degree))
                if fermat_time is not None:
                    fermat_n_values.append(n_value)
                    fermat_values.append(fermat_time)
            if fermat_values:
                all_n_values.update(fermat_n_values)
                all_y_values.extend(fermat_values)
                ax.plot(fermat_n_values, fermat_values, METHOD_STYLES['fermat'][1], label=f"{FIELD_DISPLAY_MAP['standard1']} - {METHOD_STYLES['fermat'][0]}", color=METHOD_STYLES['fermat'][2], linewidth=3, markersize=9, alpha=0.85)

    ax.set_title(f'$d={degree}$', fontweight='bold')
    ax.set_xlabel('Number of Variables (n)')
    ax.set_ylabel('Time (seconds)')
    ax.set_yscale('log', base=2)
    ax.grid(True, alpha=0.3, which='both', linewidth=0.8)
    ax.grid(True, alpha=0.15, which='minor', linewidth=0.5)

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        split = len(handles) // 2
        ax.legend(handles[:split], labels[:split], loc='upper left', framealpha=0.95)
        ax.add_artist(ax.legend(handles[split:], labels[split:], loc='lower right', framealpha=0.95))

    if all_n_values:
        set_data_xticks(ax, sorted(all_n_values))
    format_log_axis(ax, all_y_values)

    save_figure(fig, f'{output_dir}/standard_fields_d{degree}_vary_n.png')


def generate_all_plots():
    """Generate all visualization plots."""
    print('\n' + '=' * 80)
    print('GENERATING BENCHMARK VISUALIZATION PLOTS')
    print('=' * 80 + '\n')

    try:
        data = load_latest_results()
    except FileNotFoundError as error:
        print(f'Error: {error}')
        return

    output_dir = 'plots'
    Path(output_dir).mkdir(exist_ok=True)
    field_names = get_field_names(data)

    print('\n[1/5] Generating per-field comparison plots...')
    for field_key, field_name in field_names.items():
        if field_key in data['results']:
            print(f'  Processing {field_name}...')
            plot_field_comparison(data, field_key, field_name, output_dir)

    print('\n[2/5] Generating cross-field comparison plots...')
    all_n_values = set()
    for field_results in data['results'].values():
        all_n_values.update(
            int(key.split('_')[-1])
            for key in field_results.keys()
            if key.startswith('fixed_vars_')
        )

    for n_value in sorted(all_n_values):
        print(f'  Processing n={n_value}...')
        plot_cross_field_comparison(data, n_value, output_dir)

    print('\n[3/5] Generating speedup ratio plots...')
    plot_speedup_ratios(data, output_dir)

    print('\n[4/5] Generating fixed degree plots (d=2, varying n)...')
    plot_fixed_degree_comparison(data, degree=2, output_dir=output_dir)

    print('\n[5/5] Generating cross-field fixed degree comparison (d=2)...')
    plot_cross_field_fixed_degree(data, degree=2, output_dir=output_dir)

    print('\n' + '=' * 80)
    print('VISUALIZATION COMPLETE')
    print('=' * 80)
    print(f"\nAll plots saved to '{output_dir}/' directory")
    print('\nGenerated plot types:')
    print('  - Per-field comparisons (log scale)')
    print('  - Cross-field comparisons for each n value')
    print('  - Speedup ratios (Magma/drsolve) across all configurations')
    print('  - Fixed degree d=2 with varying n (per-field)')
    print('  - Fixed degree d=2 with varying n (cross-field)')
    print('=' * 80 + '\n')


if __name__ == '__main__':
    generate_all_plots()
